Figure: confidence intervals for the A/B test in table 14.1.

Left panel:  each group's CTR with its own 95% confidence interval.
Right panel: the 95% confidence interval around the *difference*
             (treatment - control), compared against zero.
"""

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Data from table 14.1 -------------------------------------------------
n_control, k_control = 1_000, 50        #A Users shown and clicks, control
n_treatment, k_treatment = 1_000, 65    #B Users shown and clicks, treatment

p_control = k_control / n_control       # 0.050
p_treatment = k_treatment / n_treatment # 0.065

z = 1.96                                #C z-value for a 95% confidence level

# --- Standard errors ------------------------------------------------------
se_control = np.sqrt(p_control * (1 - p_control) / n_control)
se_treatment = np.sqrt(p_treatment * (1 - p_treatment) / n_treatment)

# Standard error of the difference between the two proportions
se_diff = np.sqrt(
    p_control * (1 - p_control) / n_control
    + p_treatment * (1 - p_treatment) / n_treatment
)                                       #D SE_diff ~ 0.0104

diff = p_treatment - p_control          # +0.015 (the observed lift)
ci_diff = (diff - z * se_diff, diff + z * se_diff)

print(f"SE_diff = {se_diff:.4f}")
print(f"95% CI for the difference: "
      f"[{ci_diff[0] * 100:+.2f}%, {ci_diff[1] * 100:+.2f}%]")

# --- Plot -----------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(9, 4), gridspec_kw={"width_ratios": [1, 1]}
)

# Panel 1: per-group CTRs with 95% confidence intervals
groups = ["Control", "Treatment"]
rates = [p_control, p_treatment]
errors = [z * se_control, z * se_treatment]
x = [0, 1]

ax1.errorbar(
    x, rates, yerr=errors, fmt="o", color="black",
    capsize=6, markersize=7, elinewidth=1.5,
)
for xi, rate, err in zip(x, rates, errors):
    ax1.annotate(f"{rate * 100:.1f}%", (xi, rate),
                 textcoords="offset points", xytext=(10, -3), fontsize=10)
    ax1.annotate(f"[{(rate - err) * 100:.1f}%, {(rate + err) * 100:.1f}%]",
                 (xi, rate + err), textcoords="offset points",
                 xytext=(0, 6), ha="center", fontsize=8, color="dimgray")

ax1.set_xlim(-0.6, 1.6)
ax1.set_xticks(x)
ax1.set_xticklabels(groups)
ax1.set_ylabel("Click-through rate")
ax1.set_title("(a) CTR per group, 95% CI")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v * 100:.0f}%")
ax1.grid(axis="y", linestyle=":", alpha=0.5)

# Panel 2: the difference and its 95% confidence interval
ax2.axhline(0, color="black", linewidth=1, linestyle="--")   #E Zero = no effect
ax2.errorbar(
    [0], [diff], yerr=[[z * se_diff], [z * se_diff]], fmt="o",
    color="black", capsize=6, markersize=7, elinewidth=1.5,
)
ax2.annotate(f"+{diff * 100:.1f}%", (0, diff),
             textcoords="offset points", xytext=(12, -3), fontsize=10)
ax2.annotate(f"{ci_diff[1] * 100:+.2f}%", (0, ci_diff[1]),
             textcoords="offset points", xytext=(12, -3),
             fontsize=9, color="dimgray")
ax2.annotate(f"{ci_diff[0] * 100:+.2f}%", (0, ci_diff[0]),
             textcoords="offset points", xytext=(12, -3),
             fontsize=9, color="dimgray")
ax2.annotate("no difference", (0.35, 0), textcoords="offset points",
             xytext=(0, 5), fontsize=9, color="dimgray")

ax2.set_xlim(-0.6, 1.0)
ax2.set_xticks([0])
ax2.set_xticklabels(["Treatment − Control"])
ax2.set_ylabel("Difference in CTR")
ax2.set_title("(b) Difference, 95% CI")
ax2.yaxis.set_major_formatter(lambda v, _: f"{v * 100:+.0f}%")
ax2.grid(axis="y", linestyle=":", alpha=0.5)

fig.tight_layout()
fig.savefig("ab_test_confidence_intervals.png", dpi=300,
            bbox_inches="tight")
plt.show()